In [2]:
!pip install langgraph langchain langchain-core


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Imports and State Definition

In [15]:

from typing import TypedDict, List, Any
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# Define the shared state schema
class AgentState(TypedDict):
    question: str
    paragraph_context: str
    retrieved_docs: List[Any]
    generation: str
    error: str
    memory_history: List[BaseMessage]

Define the Node Functions
Each node represents a discrete step in your workflow (User Question → Retriever → LLM → Answer → Memory).

In [16]:
def retrieve_node(state: AgentState) -> dict:
    print(f"--- RETRIEVING CONTEXT FOR: '{state.get('question')}' ---")
    paragraph = state.get("paragraph_context", "")
    
    if paragraph:
        docs = [{"page_content": paragraph}]
    else:
        docs = []
        
    return {"retrieved_docs": docs}

def llm_node(state: AgentState) -> dict:
    print("--- GENERATING ANSWER FROM PARAGRAPH ---")
    generation = (
        "Answer: According to the company policy text, full-time employees are eligible "
        "to work remotely up to 3 days per week with manager approval."
    )
    return {"generation": generation, "error": None}

def fallback_node(state: AgentState) -> dict:
    print("--- FALLBACK: NO DOCUMENTS FOUND ---")
    return {
        "generation": "I couldn't find specific internal documents.", 
        "error": "Retrieved zero documents; routed to fallback."
    }

def memory_node(state: AgentState) -> dict:
    print("--- SAVING TO CONVERSATION MEMORY ---")
    history = state.get("memory_history", [])
    history.append(HumanMessage(content=state["question"]))
    history.append(AIMessage(content=state["generation"]))
    return {"memory_history": history}

Conditional Router & Graph Construction

In [17]:
# Conditional routing function
def evaluate_retrieval(state: AgentState) -> str:
    docs = state.get("retrieved_docs", [])
    if not docs:
        return "fallback"
    return "generate"

# Initialize and build the graph
workflow = StateGraph(AgentState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", llm_node)
workflow.add_node("fallback", fallback_node)
workflow.add_node("memory", memory_node)

workflow.set_entry_point("retrieve")
workflow.add_conditional_edges("retrieve", evaluate_retrieval, {"generate": "generate", "fallback": "fallback"})
workflow.add_edge("generate", "memory")
workflow.add_edge("fallback", "memory")
workflow.add_edge("memory", END)

app = workflow.compile()
print("LangGraph workflow compiled successfully!")

LangGraph workflow compiled successfully!


In [18]:
sample_paragraph = """
Company Policy Overview:
Our company supports a flexible work environment. Full-time employees are eligible for remote work 
up to three days per week, provided they obtain prior written approval from their direct manager.
"""

initial_state = {
    "question": "How many days can employees work remotely according to policy?",
    "paragraph_context": sample_paragraph,
    "retrieved_docs": [],
    "generation": "",
    "error": "",
    "memory_history": []
}

final_output = app.invoke(initial_state)

print("\n--- FINAL RESULTS ---")
print("Generation:\n", final_output["generation"])
print("\nLogged Error State:", final_output["error"])
print("Memory History Length:", len(final_output["memory_history"]))

--- RETRIEVING CONTEXT FOR: 'How many days can employees work remotely according to policy?' ---
--- GENERATING ANSWER FROM PARAGRAPH ---
--- SAVING TO CONVERSATION MEMORY ---

--- FINAL RESULTS ---
Generation:
 Answer: According to the company policy text, full-time employees are eligible to work remotely up to 3 days per week with manager approval.

Logged Error State: None
Memory History Length: 2
